# Testing workflow for IRTransport

In [ ]:
# # Dependency nightmare with this repo ... 

# mamba create -n irtransport -c conda-forge -y \
#   python=3.11 numpy=1.26 scipy=1.12 pot=0.9.4 tqdm ipykernel rust pip
# mamba activate irtransport

# pip install --no-deps "polars==0.23.4"

In [1]:
from ir_transport import IRTransport
import polars as pl
from ir_transport.ir_transport import IRTransport

In [ ]:
# import polars, ir_transport, importlib.metadata as im

# print("✓ ready →", {
#     "polars": polars.__version__,
#     "ir_transport": getattr(ir_transport, "__version__", "dev"),
#     "tcrdist_rs": im.version("tcrdist-rs")
# })

Trying with our own data

In [4]:
import polars as pl
from ir_transport.ir_transport import IRTransport

FILE1 = "/central/groups/MazmanianLab/joeB/PDMBS/parkinsons-microbial-blood-signatures/data/interim/irtransport/irt_input/PD-PDWB481MXU-BLM0T1.tsv"
FILE2 = "/central/groups/MazmanianLab/joeB/PDMBS/parkinsons-microbial-blood-signatures/data/interim/irtransport/irt_input/PP-4021-BLM0T1.tsv"
df1 = pl.read_csv(FILE1, separator="\t")
df2 = pl.read_csv(FILE2, separator="\t")

# df1.head()
# df2.head()

# ------------------------------------------------------------------
# 1️⃣  one-time concat patch  (skips if already applied)
# ------------------------------------------------------------------
if not getattr(pl, "_concat_patched", False):

    def _cast_uints(df):
        uints = [c for c, t in zip(df.columns, df.dtypes)
                 if t in (pl.UInt8, pl.UInt16, pl.UInt32, pl.UInt64)]
        return df.with_columns(pl.col(uints).cast(pl.Int64)) if uints else df

    _concat_orig = pl.concat

    def _concat_fix(elems, *args, **kw):
        elems = [_cast_uints(d) for d in elems]
        return _concat_orig(elems, *args, **kw)

    pl.concat           = _concat_fix
    pl._concat_patched  = True        # marker so we don’t double-wrap


# ------------------------------------------------------------------
# 1️⃣  patch for drop
# ------------------------------------------------------------------
# save original method
_orig_drop = pl.DataFrame.drop
def _drop_expr_safe(self, columns, *args, **kwargs):
    """
    Extend pl.DataFrame.drop() so it also accepts an Expr/regex selector
    (feature present in Polars ≥0.26, missing in 0.20.x).
    """
    # If the user passed an Expr, evaluate it to concrete column names.
    if isinstance(columns, pl.Expr):
        columns = self.select(columns).columns     # turn Expr into list[str]
    return _orig_drop(self, columns, *args, **kwargs)

# patch once
if not getattr(pl, "_drop_patched", False):
    pl.DataFrame.drop = _drop_expr_safe
    pl._drop_patched = True


# ------------------------------------------------------------------
# 2️⃣  run IRTransport as normal
# ------------------------------------------------------------------
irt = IRTransport()
irt.add_dataset(df1, seq_cols=["cdr3b", "vb"], reference=True)
irt.add_dataset(df2, seq_cols=["cdr3b", "vb"])

irt.compute_enrichment()
irt.compute_significance()      # should now finish
irt.create_clusters()


Randomizing datasets and computing enrichment:  23%|██▎       | 23/100 [06:49<08:10,  6.37s/it]  /central/groups/MazmanianLab/joeB/software/mambaforge/envs/irtransport/lib/python3.11/site-packages/ot/bregman/_sinkhorn.py:531: UserWarning: Sinkhorn did not converge. You might want to increase the number of iterations `numItermax` or the regularization parameter `reg`.
  warnings.warn("Sinkhorn did not converge. You might want to "
Creating clusters: 100%|██████████| 10/10 [00:09<00:00,  1.01it/s]


(shape: (398, 13)
 ┌───────┬──────────┬─────────────┬─────────────┬───┬───────────┬─────────┬────────────┬────────────┐
 │ index ┆ vb       ┆ cdr3b       ┆ multiplicit ┆ … ┆ z_score   ┆ p_value ┆ cluster_ra ┆ transport_ │
 │ ---   ┆ ---      ┆ ---         ┆ y           ┆   ┆ ---       ┆ ---     ┆ dius       ┆ cluster    │
 │ i64   ┆ str      ┆ str         ┆ ---         ┆   ┆ f64       ┆ f64     ┆ ---        ┆ ---        │
 │       ┆          ┆             ┆ i64         ┆   ┆           ┆         ┆ f64        ┆ i32        │
 ╞═══════╪══════════╪═════════════╪═════════════╪═══╪═══════════╪═════════╪════════════╪════════════╡
 │ 0     ┆ TRBV10-3 ┆ CAIRAGLADEQ ┆ 1           ┆ … ┆ 1.182992  ┆ 0.15    ┆ null       ┆ null       │
 │       ┆          ┆ YF          ┆             ┆   ┆           ┆         ┆            ┆            │
 │ 1     ┆ TRBV10-3 ┆ CAIRATGRRNS ┆ 1           ┆ … ┆ -0.815884 ┆ 0.75    ┆ null       ┆ null       │
 │       ┆          ┆ PLHF        ┆             ┆   ┆           

In [8]:
base_cols = irt.seq_cols + ["multiplicity"]
stat_cols = ["enrichment", "num_neighbors",
             "p_value", "p_value_bh",          # or whatever suffix you chose
             "transport_cluster", "cluster_radius"]

# results = 
irt.df_samp
# results

index,vb,cdr3b,multiplicity,enrichment,num_neighbors,tmp_enrichment,tmp_num_neighbors,scores,z_score,p_value,cluster_radius,transport_cluster
i64,str,str,i64,f64,i64,f64,i64,list[f64],f64,f64,f64,i32
0,"""TRBV10-3""","""CAIRAGLADEQYF""",1,132.18824,1,64.775108,1,"[53.463438, 53.534552, … 218.031033]",1.182992,0.15,null,null
1,"""TRBV10-3""","""CAIRATGRRNSPLHF""",1,79.452283,0,0.0,0,"[76.414659, 76.641085, … 98.853927]",-0.815884,0.75,null,null
2,"""TRBV10-3""","""CAISDPGGSSYEQYF""",1,125.197988,1,70.637775,1,"[42.631876, 42.639951, … 335.449988]",0.066026,0.44,null,null
3,"""TRBV10-3""","""CAISDPTSGVGDTQYF""",1,74.338197,0,0.0,0,"[71.691481, 71.874614, … 100.439795]",-0.744227,0.77,null,null
4,"""TRBV10-3""","""CAISDVGSGPDYEQYF""",1,125.197988,1,54.560214,1,"[50.113359, 50.153239, … 252.19814]",0.584387,0.26,null,null
…,…,…,…,…,…,…,…,…,…,…,…,…
393,"""TRBV29-1""","""CSVTISGGSNQPQHF""",1,79.45474,0,0.0,0,"[63.062992, 63.221768, … 97.748198]",0.407125,0.35,null,null
394,"""TRBV29-1""","""CSVVIGGSYNEQFF""",1,125.631102,1,57.654344,1,"[50.470491, 50.500854, … 218.574687]",0.752786,0.26,null,null
395,"""TRBV20-1""","""FSARATPGRVGPLHF""",1,154.679852,1,77.339926,1,"[49.352409, 49.844986, … 196.761146]",1.790465,0.19,null,null


In [16]:
# irt.df_samp.write_csv("df_samp_full.csv")
irt.df_samp.drop("scores").write_csv("df_samp_full.csv")